# convert 명령 테스트 (작업지시서 §3)

지금까지 만든 부품(방출기 4종 + 캐시)을 **한 명령으로 조립**하는 게 `cli.py`입니다.
`convert`가 하는 일 한 줄: **원본 하나 → `converted/{stem_slug}_{sha12}/` 패키지 폴더**.

이번 1차 범위(M1, 갈래 1): `meta.json` + `data/{cells.jsonl, references.json,
diagnostics.json}`. `SKILL.md`·`layout/*.html`은 방출기가 아직 없어(M2) 제외.
`annotate`/`review`/`verify`는 등록만 된 스텁입니다.

조립 계약(오너 확정):
- **stdout은 패키지 경로만** — 진행·cache 사유·경고·오류는 전부 stderr(스크립트 연계).
- **generated_at은 한 변환에서 1회** — meta.json과 _index.json에 같은 값.
- **원자적 생성** — 임시 폴더에 다 쓰고 성공해야 최종 폴더로 교체 + 그 뒤에야 색인 기록.
  실패하면 반쪽 폴더가 hit로 안 잡힘.
- **hit이면 아무 파일도 다시 안 씀** — 기존 경로만 출력.

확인하는 것:
1. **첫 변환(miss)** — 폴더·4산출물·색인이 생기고 stdout은 경로 한 줄뿐
2. **재변환(hit)** — 파일을 다시 안 씀(mtime 불변), 경로만 출력
3. **--force** — hit 무시하고 재생성(mtime 바뀜)
4. **결정론** — 다른 출력 루트 두 곳에 변환 → generated_at 빼면 동일
5. **--all** — 디렉터리 일괄, 성공마다 경로 한 줄
6. **스텁·플래그 안내** — verify/annotate/review는 exit 2, 미지원 플래그는 stderr 고지

In [1]:
# 0. 준비 — subprocess로 실제 CLI를 돌린다
import json, os, shutil, subprocess, sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

RAW = ROOT / "workingpaper_raw"
SANDBOX = ROOT / "tests" / "_output" / "cli_nb"
if SANDBOX.exists():
    shutil.rmtree(SANDBOX)
SANDBOX.mkdir(parents=True)

ENV = {**os.environ, "PYTHONPATH": str(ROOT / "src")}

def run(*cli_args):
    """CLI를 자식 프로세스로 실행 → (rc, stdout, stderr)."""
    p = subprocess.run(
        [sys.executable, "-m", "excel_to_skill.cli", *cli_args],
        cwd=str(ROOT), env=ENV, capture_output=True, text=True)
    return p.returncode, p.stdout, p.stderr

SRC = next(RAW.glob("*1100~1300*"))
print("루트:", ROOT)
print("샘플 원본:", SRC.name)

루트: /home/shin/Project/Excel_to_Skill
샘플 원본: 감사조서서식_1100~1300 감사계약.xlsx


## 1. 첫 변환(miss) — 폴더·4산출물·색인 + stdout 순도

원본 하나를 변환합니다. 기대: exit 0, **stdout은 패키지 경로 딱 한 줄**(로그는 stderr로
가야 함), 폴더 안에 `meta.json`과 `data/`의 3파일, 그리고 `_index.json`에 1항목.

In [2]:
OUT = SANDBOX / "c1"
rc, out, err = run("convert", str(SRC), "--out", str(OUT), "--no-annotate")
print("exit:", rc)
print("stdout:", repr(out))
print("stderr:", err.strip())

pkg = Path(out.strip())
assert rc == 0
# stdout 순도: 딱 한 줄, 그게 실존 폴더
lines = [l for l in out.splitlines() if l.strip()]
assert len(lines) == 1 and pkg.is_dir(), "stdout은 경로 한 줄이어야"
assert "cache miss:absent" in err     # 사유는 stderr로
# 4산출물
for rel in ["meta.json", "data/cells.jsonl", "data/references.json", "data/diagnostics.json"]:
    assert (pkg / rel).is_file(), rel
    print("  O", rel)
# 색인 1항목
idx = json.loads((OUT / "_index.json").read_text(encoding="utf-8"))
assert len(idx["entries"]) == 1
print("색인 항목 수:", len(idx["entries"]))
# meta.json과 _index.json의 generated_at 동일(한 변환=한 시각)
meta_gen = json.loads((pkg / "meta.json").read_text(encoding="utf-8"))["generated_at"]
entry_gen = next(iter(idx["entries"].values()))["generated_at"]
print("generated_at (meta==index):", meta_gen == entry_gen, meta_gen)
assert meta_gen == entry_gen
print("\n결과: 통과")

exit: 0
stdout: '/home/shin/Project/Excel_to_Skill/tests/_output/cli_nb/c1/감사조서서식_1100~1300_감사계약_9c7aa730172a\n'
stderr: [cache miss:absent] 감사조서서식_1100~1300 감사계약.xlsx → 생성
  O meta.json
  O data/cells.jsonl
  O data/references.json
  O data/diagnostics.json
색인 항목 수: 1
generated_at (meta==index): True 2026-07-09T04:44:48Z

결과: 통과


## 2. 재변환(hit) — 아무 파일도 다시 쓰지 않는다

같은 원본을 한 번 더. 기대: exit 0, stdout은 같은 경로, stderr에 `cache hit`.
**핵심**: 파일을 다시 쓰지 않으므로 `meta.json`의 수정시각(mtime)이 그대로여야 합니다.

In [3]:
before = (pkg / "meta.json").stat().st_mtime_ns
rc, out, err = run("convert", str(SRC), "--out", str(OUT), "--no-annotate")
after = (pkg / "meta.json").stat().st_mtime_ns
print("exit:", rc, "| stderr:", err.strip())
print("stdout 경로 동일:", out.strip() == str(pkg))
print("meta.json 미변경(mtime 동일):", before == after)
assert rc == 0 and out.strip() == str(pkg)
assert "cache hit" in err
assert before == after, "hit인데 파일이 다시 써짐"
print("\n결과: 통과 (hit = 무재작성)")

exit: 0 | stderr: [cache hit] 감사조서서식_1100~1300 감사계약.xlsx → 감사조서서식_1100~1300_감사계약_9c7aa730172a (재생성 생략)
stdout 경로 동일: True
meta.json 미변경(mtime 동일): True

결과: 통과 (hit = 무재작성)


## 3. --force — hit을 무시하고 재생성

`--force`는 캐시를 무시하고 다시 만듭니다. 기대: stderr `cache miss:force`,
`meta.json` mtime이 **바뀜**, stdout은 같은 경로.

In [4]:
before = (pkg / "meta.json").stat().st_mtime_ns
rc, out, err = run("convert", str(SRC), "--out", str(OUT), "--no-annotate", "--force")
after = (pkg / "meta.json").stat().st_mtime_ns
print("exit:", rc, "| stderr:", err.strip())
assert rc == 0 and out.strip() == str(pkg)
assert "cache miss:force" in err
assert before != after, "force인데 재생성 안 됨"
print("meta.json 재작성됨(mtime 변경):", before != after)
print("\n결과: 통과")

exit: 0 | stderr: [cache miss:force] 감사조서서식_1100~1300 감사계약.xlsx → 생성
meta.json 재작성됨(mtime 변경): True

결과: 통과


## 4. 결정론 — 서로 다른 출력 루트, 같은 결과

같은 원본을 빈 루트 두 곳에 각각 변환합니다. `data/`의 3파일은 **바이트까지 동일**,
`meta.json`은 `generated_at`만 빼면 동일해야 합니다(V3 재현성의 명령 수준 확인).

In [5]:
A = SANDBOX / "detA"; B = SANDBOX / "detB"
rcA, outA, _ = run("convert", str(SRC), "--out", str(A), "--no-annotate")
rcB, outB, _ = run("convert", str(SRC), "--out", str(B), "--no-annotate")
pa, pb = Path(outA.strip()), Path(outB.strip())
assert rcA == 0 and rcB == 0

for rel in ["data/cells.jsonl", "data/references.json", "data/diagnostics.json"]:
    same = (pa / rel).read_bytes() == (pb / rel).read_bytes()
    print(f"  {'O' if same else 'X'} {rel} 바이트 동일")
    assert same, rel

def meta_norm(p):
    d = json.loads((p / "meta.json").read_text(encoding="utf-8"))
    d.pop("generated_at")
    return d
print("  meta.json (generated_at 제외) 동일:", meta_norm(pa) == meta_norm(pb))
assert meta_norm(pa) == meta_norm(pb)
print("\n결과: 통과 (결정론)")

  O data/cells.jsonl 바이트 동일
  O data/references.json 바이트 동일
  O data/diagnostics.json 바이트 동일
  meta.json (generated_at 제외) 동일: True

결과: 통과 (결정론)


## 5. --all — 디렉터리 일괄 변환

디렉터리 최상위의 `*.xlsx`/`*.xls`를 정렬 순회합니다(재귀 안 함, `~$` 제외).
코퍼스에서 작은 파일 3개를 임시 폴더에 복사해 일괄 변환하고, **성공마다 경로 한 줄**
(stdout N줄)과 색인 N항목을 확인합니다.

In [6]:
INBOX = SANDBOX / "inbox"; INBOX.mkdir()
picks = sorted(RAW.glob("*.xls*"), key=lambda f: f.stat().st_size)[:3]
for f in picks:
    shutil.copy(f, INBOX / f.name)
print("복사한 원본:", [f.name for f in picks])

OUT_ALL = SANDBOX / "call"
rc, out, err = run("convert", "--all", str(INBOX), "--out", str(OUT_ALL), "--no-annotate")
paths = [l for l in out.splitlines() if l.strip()]
print("exit:", rc)
print("stdout 줄 수:", len(paths))
print(err.strip().splitlines()[-1])  # [요약] 줄
assert rc == 0 and len(paths) == 3
assert all(Path(p).is_dir() for p in paths)
idx = json.loads((OUT_ALL / "_index.json").read_text(encoding="utf-8"))
print("색인 항목 수:", len(idx["entries"]))
assert len(idx["entries"]) == 3
print("\n결과: 통과 (--all N건 = stdout N줄 = 색인 N항목)")

복사한 원본: ['감사조서서식_04 감사조서철 작성 및 보존.xlsx', '감사조서서식_8550A_업무수행이사와 논의한 사항 및 자문한 사항 2025 신규.xlsx', '감사조서서식_7555A_그룹감사 업무수행이사와 논의한 사항 및 자문한 사항 2025 신규.xlsx']


exit: 0
stdout 줄 수: 3
[요약] 3건 중 성공 3 · 실패 0
색인 항목 수: 3

결과: 통과 (--all N건 = stdout N줄 = 색인 N항목)


## 6. 스텁과 플래그 안내

- `verify`/`annotate`/`review`는 등록만 된 스텁 → **exit 2** + stderr 안내.
- 미지원·무의미 플래그(`--full-names`, `--max-rows`, `--force-annotate`, `--model`,
  그리고 `--no-annotate` 생략)는 **조용히 무시하지 않고** stderr로 고지.

In [7]:
# 스텁 3종
for cmd in ("verify", "annotate", "review"):
    rc, out, err = run(cmd, "x")
    print(f"  {cmd:8} exit={rc}  '{err.strip()}'")
    assert rc == 2 and "구현되지 않" in err

# 플래그 안내 (별도 루트, 실제 생성은 되게)
OUT_W = SANDBOX / "warn"
# --no-annotate 생략 → 주석기 미구현 안내
_, _, err = run("convert", str(SRC), "--out", str(OUT_W))
assert "주석기" in err
print("\n  O --no-annotate 생략 → 주석기 미구현 고지")
# --full-names / --max-rows 고지 (이미 hit이라 생성은 생략됨, 안내는 나와야)
_, _, err = run("convert", str(SRC), "--out", str(OUT_W), "--no-annotate",
                "--full-names", "--max-rows", "10")
assert "--full-names" in err and "--max-rows" in err
print("  O --full-names / --max-rows → 미적용 고지")
print("\n결과: 통과")

  verify   exit=2  ''verify' 은(는) 아직 구현되지 않았습니다 (M3/verify 단계).'


  annotate exit=2  ''annotate' 은(는) 아직 구현되지 않았습니다 (M3/verify 단계).'


  review   exit=2  ''review' 은(는) 아직 구현되지 않았습니다 (M3/verify 단계).'



  O --no-annotate 생략 → 주석기 미구현 고지
  O --full-names / --max-rows → 미적용 고지

결과: 통과
